In [58]:
import modal

In [ ]:


import os
from codegen.configs.models.secrets import SecretsConfig
from codegen.sdk.core.codebase import Codebase
from codegen.shared.enums.programming_language import ProgrammingLanguage

class CodebaseSnapshot(modal.Cls):
    codebase: Codebase
    repo_org: str | None = modal.parameter(default="codegen-sh")   
    repo_name: str | None = modal.parameter(default="codegen-sdk")
    commit: str | None = modal.parameter(default=None)
    temp_dir: str = "/root"


    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
    
    @modal.enter(snap=True)
    def load(self):
        config = SecretsConfig()
        config.github_token = os.environ["GITHUB_TOKEN"]
        self.codebase = Codebase.from_repo(f"{self.repo_org}/{self.repo_name}", commit=self.commit, language=ProgrammingLanguage.PYTHON, tmp_dir=self.temp_dir, secrets=config)


In [38]:

image = modal.Image.debian_slim(python_version="3.13").apt_install("git").pip_install("fastapi[standard]", "codegen==0.30.1")


app = modal.App("snapshot-test", image=image)


In [25]:


@app.cls(secrets=[modal.Secret.from_dotenv()], enable_memory_snapshot=True)
class SubClassTest(CodebaseSnapshot):
    


    @modal.method()
    def files(self):
        return len(self.codebase.files)




In [59]:
Klass = modal.Cls.from_name(app_name="snapshot-test", name="SubClassTest")
klass = Klass(repo_org="codegen-sh", repo_name="codegen-sdk", commit="834b600c6802f9eb33023cb30abeb7a96c80f495")

In [60]:
klass.files.remote()

KeyboardInterrupt: 

In [42]:
Klass = modal.Cls.from_name(app_name="snapshot-test", name="SubClassTest")
klass = Klass(repo_org="codegen-sh", repo_name="codegen-sdk", commit="eb7ca7445c2afd580eaf94a87c4f0dbe281463ee")


In [ ]:
klass.files.remote()

In [52]:
c = Codebase.from_repo(commit="eb7ca7445c2afd580eaf94a87c4f0dbe281463ee", repo_full_name="codegen-sh/codegen-sdk")

In [ ]:
len(c.files)